# ⚡ PETA KILAT & MASTER CHEATSHEET RESPONSI UTS PRAKTIKUM BIG DATA
> **Notebook Siap Main (Playable):** Notebook ini adalah konversi langsung dari README utama. Semua blok kode di bawah ini **bisa kamu jalankan satu per satu (Shift + Enter)** langsung di Jupyter Notebook/Lab untuk melihat cara kerjanya secara nyata!

## 🧠 The Big Picture: Cerita Sederhana 6 Pertemuan (Biar Langsung "Klik")

```
[Pertemuan 2: Masa Lalu]
Data masih kecil -> Pake Pandas di laptop -> Gampang & cepat.
Tapi begitu data capai 100 GB -> Laptop crash (MemoryError: RAM habis)!
                         │
                         ▼
[Pertemuan 3: Tempat Penyimpanan Data Raksasa (Storage)]
Solusinya: Data dipecah-pecah ke banyak komputer -> Lahirlah APACHE HADOOP (HDFS).
HDFS ibarat gudang raksasa: NameNode = mandor (tahu lokasi barang), DataNode = kuli/rak (nyimpen barang fisik).
                         │
                         ▼
[Pertemuan 4: Otak Komputasi Cepat (Engine)]
Hadoop punya mesin hitung bernama MapReduce, tapi LAMBAT karena bolak-balik nulis ke harddisk.
Lalu datang APACHE SPARK: Menghitung langsung di RAM (In-Memory), 100x lebih kencang!
Kita pakai PySpark DataFrame dengan sifat "Lazy Evaluation" (tidak langsung hitung sebelum diminta aksi).
                         │
                         ▼
[Pertemuan 5: Operasi Pengolahan Canggih]
Data di dunia nyata terpisah-pisah di banyak tabel:
1. Kita gabungkan tabel (JOIN).
2. Kita cari ranking terbaik per kategori (WINDOW FUNCTION).
3. Kita hitung pakai gaya query database biasa (SPARK SQL).
                         │
                         ▼
[Pertemuan 6: Format File Masa Depan & Pipeline ETL]
Simpan data jangan pakai CSV (lambat & boros ruang)!
Pakai format PARQUET (berorientasi kolom, terkompres Snappy otomatis, hemat 80% storage).
Lalu kita rapihkan jadi alur resmi: EXTRACT (baca) -> TRANSFORM (bersihkan & hitung) -> LOAD (simpan).
                         │
                         ▼
[Pertemuan 7: Gladi Bersih Ujian / Studi Kasus Lengkap]
Semua digabung jadi 1 alur utuh: Ingest ke HDFS -> Extract pakai Spark -> Transform (Join + Window + SQL) -> Load ke Parquet HDFS -> Berikan kesimpulan bisnis ke manajemen!
```

## 🚀 Alur Sakti Ujian Responsi (Checklist 5 Langkah)

1. **Nyalakan Klaster Hadoop di Terminal:**
   ```bash
   sudo systemctl start sshd
   start-dfs.sh
   start-yarn.sh
   jps
   ```
2. **Buka Jupyter Lab di Repositori:**
   ```bash
   cd /mnt/alterra/repos/praktikum-big-data
   uv run jupyter lab
   ```
3. **Unggah Berkas ke HDFS:**
   `hdfs dfs -mkdir -p /user/$USER/responsi/raw && hdfs dfs -put -f file.csv /user/$USER/responsi/raw/`
4. **Jalankan Template PySpark (Di bawah ini):**
5. **Bersihkan Klaster Setelah Selesai:**
   `stop-yarn.sh && stop-dfs.sh && sudo systemctl stop sshd`

### 0. Uji Status Klaster Hadoop di Laptop
Jalankan sel berikut untuk memastikan daemon Hadoop berjalan jika kamu ingin menguji HDFS:

In [1]:
# Cek daemon Hadoop yang aktif
!jps

69097 Jps


## 📋 Master Code Cheatsheet (Siap Jalan & Diuji)

### 1. Inisialisasi SparkSession (Arch Linux Zero-Conflict Safe)
Menyiapkan variabel lingkungan agar PySpark menggunakan Java 11 terisolasi dan binding IP 127.0.0.1:

In [2]:
import os
import sys
import getpass
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Konfigurasi isolasi lingkungan Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Buat SparkSession
spark = SparkSession.builder \
    .appName("Responsi-UTS-MasterCheatSheet") \
    .master("local[*]") \
    .getOrCreate()

# Supaya sel bersih dari spam log info
spark.sparkContext.setLogLevel("ERROR")
username = os.environ.get("USER", getpass.getuser())
print(f"✅ SparkSession Berhasil Aktif untuk {username}!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/01 11:29:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession Berhasil Aktif untuk kaky!


### 2. Membaca Data (EXTRACT)
Di bawah ini kita siapkan data simulasi transaksi retail dan master cabang langsung di dalam memori agar kamu bisa langsung mengujinya tanpa tergantung berkas eksternal:

In [3]:
# Data simulasi transaksi e-commerce multi-cabang
data_transaksi = [
    ("TRX-001", "Magelang", "Elektronik", 2, 750000, 5),
    ("TRX-002", "Magelang", "Fashion", 5, 120000, None),
    ("TRX-003", "Yogyakarta", "Elektronik", 1, 1500000, 4),
    ("TRX-004", "Yogyakarta", "Makanan", 15, 25000, 5),
    ("TRX-005", "Semarang", "Elektronik", 3, 600000, None),
    ("TRX-006", "Semarang", "Fashion", 4, 150000, 3),
    ("TRX-007", "Purworejo", "Makanan", 20, 15000, 4),
    ("TRX-008", "Purworejo", "Elektronik", 1, 2000000, None),
    ("TRX-009", "Magelang", "Makanan", 10, 30000, 5),
    ("TRX-010", "Solo", "Fashion", 6, 110000, 4),
]
cols_trx = ["order_id", "kota", "kategori", "unit_terjual", "harga_satuan", "rating"]
df_trx = spark.createDataFrame(data_transaksi, schema=cols_trx)

# Data tabel target cabang
data_cabang = [
    ("Magelang", "Rani Handayani", 2500000),
    ("Yogyakarta", "Joko Widodo", 4000000),
    ("Semarang", "Sari Indah", 3000000),
    ("Purworejo", "Fitri Wulandari", 2000000),
    ("Solo", "Bayu Nugroho", 1500000),
]
df_cabang = spark.createDataFrame(data_cabang, schema=["kota", "pic_cabang", "target_omset"])

print("Tabel Transaksi:")
df_trx.show(5)
print("Tabel Master Cabang:")
df_cabang.show()

Tabel Transaksi:


+--------+----------+----------+------------+------------+------+
|order_id|      kota|  kategori|unit_terjual|harga_satuan|rating|
+--------+----------+----------+------------+------------+------+
| TRX-001|  Magelang|Elektronik|           2|      750000|     5|
| TRX-002|  Magelang|   Fashion|           5|      120000|  NULL|
| TRX-003|Yogyakarta|Elektronik|           1|     1500000|     4|
| TRX-004|Yogyakarta|   Makanan|          15|       25000|     5|
| TRX-005|  Semarang|Elektronik|           3|      600000|  NULL|
+--------+----------+----------+------------+------------+------+
only showing top 5 rows

Tabel Master Cabang:


+----------+---------------+------------+
|      kota|     pic_cabang|target_omset|
+----------+---------------+------------+
|  Magelang| Rani Handayani|     2500000|
|Yogyakarta|    Joko Widodo|     4000000|
|  Semarang|     Sari Indah|     3000000|
| Purworejo|Fitri Wulandari|     2000000|
|      Solo|   Bayu Nugroho|     1500000|
+----------+---------------+------------+



### 3. Operasi Dasar DataFrame (TRANSFORM Bagian 1)
- `select()` untuk memilih kolom
- `filter()` kondisi ganda
- `withColumn()` untuk menghitung total pendapatan
- `when().otherwise()` untuk pengelompokan kondisi
- `na.fill()` untuk imputasi nilai null

In [4]:
# 1. Pilih kolom tertentu
df_pilih = df_trx.select("order_id", "kota", "unit_terjual", "harga_satuan")

# 2. Tambah kolom total pendapatan (unit x harga)
df_olah = df_trx.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga_satuan"))

# 3. Kondisi IF-ELSE di PySpark: when dan otherwise
df_olah = df_olah.withColumn(
    "status_transaksi",
    F.when(F.col("total_pendapatan") >= 1000000, "Transaksi Besar")
     .otherwise("Transaksi Reguler")
)

# 4. Deteksi null & imputasi nilai default (sentinel value 0)
df_olah = df_olah.withColumn("ada_rating", F.col("rating").isNotNull())
df_bersih = df_olah.na.fill({"rating": 0})

# 5. Filter baris dengan kondisi ganda (& / |)
df_filter = df_bersih.filter((F.col("total_pendapatan") >= 500000) & (F.col("kota") != "Solo"))

print("Hasil Transformasi & Filter:")
df_filter.show()

Hasil Transformasi & Filter:


+--------+----------+----------+------------+------------+------+----------------+-----------------+----------+
|order_id|      kota|  kategori|unit_terjual|harga_satuan|rating|total_pendapatan| status_transaksi|ada_rating|
+--------+----------+----------+------------+------------+------+----------------+-----------------+----------+
| TRX-001|  Magelang|Elektronik|           2|      750000|     5|         1500000|  Transaksi Besar|      true|
| TRX-002|  Magelang|   Fashion|           5|      120000|     0|          600000|Transaksi Reguler|     false|
| TRX-003|Yogyakarta|Elektronik|           1|     1500000|     4|         1500000|  Transaksi Besar|      true|
| TRX-005|  Semarang|Elektronik|           3|      600000|     0|         1800000|  Transaksi Besar|     false|
| TRX-006|  Semarang|   Fashion|           4|      150000|     3|          600000|Transaksi Reguler|      true|
| TRX-008| Purworejo|Elektronik|           1|     2000000|     0|         2000000|  Transaksi Besar|    

### 4. Relational Join (TRANSFORM Bagian 2)
**Best Practice Ujian:** Ringkas (agregasi) tabel transaksi besar terlebih dahulu, baru kemudian di-join dengan tabel target cabang untuk mencegah *network shuffle*!

In [5]:
# Agregasi omset transaksi per kota
df_omset_kota = df_bersih.groupBy("kota").agg(
    F.sum("total_pendapatan").alias("realisasi_omset")
)

# Inner Join dengan tabel target cabang
df_performa = df_omset_kota.join(df_cabang, on="kota", how="inner") \
    .withColumn("persen_capaian", F.round((F.col("realisasi_omset") / F.col("target_omset")) * 100, 2)) \
    .orderBy(F.col("persen_capaian").desc())

print("Hasil Analisis Pencapaian Target Cabang:")
df_performa.show()

Hasil Analisis Pencapaian Target Cabang:


+----------+---------------+---------------+------------+--------------+
|      kota|realisasi_omset|     pic_cabang|target_omset|persen_capaian|
+----------+---------------+---------------+------------+--------------+
| Purworejo|        2300000|Fitri Wulandari|     2000000|         115.0|
|  Magelang|        2400000| Rani Handayani|     2500000|          96.0|
|  Semarang|        2400000|     Sari Indah|     3000000|          80.0|
|Yogyakarta|        1875000|    Joko Widodo|     4000000|         46.88|
|      Solo|         660000|   Bayu Nugroho|     1500000|          44.0|
+----------+---------------+---------------+------------+--------------+



### 5. Window Function (TRANSFORM Bagian 3: Ranking Top-N)
Mencari produk / kategori terlaris (Juara 1) di setiap kota tanpa menghilangkan kelompok baris:

In [6]:
# Langkah A: Agregasi total omset per kombinasi kota & kategori
df_kota_kategori = df_bersih.groupBy("kota", "kategori").agg(
    F.sum("total_pendapatan").alias("total_omset_kategori")
)

# Langkah B: Definisikan window partisi per kota urut omset menurun
window_kota = Window.partitionBy("kota").orderBy(F.col("total_omset_kategori").desc())

# Langkah C: Beri nomor urut (row_number) dan filter peringkat 1
df_top1 = df_kota_kategori \
    .withColumn("peringkat", F.row_number().over(window_kota)) \
    .filter(F.col("peringkat") == 1) \
    .orderBy(F.col("total_omset_kategori").desc())

print("Kategori Terlaris (Juara 1) di Setiap Kota:")
df_top1.show()

Kategori Terlaris (Juara 1) di Setiap Kota:


+----------+----------+--------------------+---------+
|      kota|  kategori|total_omset_kategori|peringkat|
+----------+----------+--------------------+---------+
| Purworejo|Elektronik|             2000000|        1|
|  Semarang|Elektronik|             1800000|        1|
|  Magelang|Elektronik|             1500000|        1|
|Yogyakarta|Elektronik|             1500000|        1|
|      Solo|   Fashion|              660000|        1|
+----------+----------+--------------------+---------+



### 6. Spark SQL (TRANSFORM Bagian 4: Kueri SQL Murni)
Menggunakan SQL deklaratif melalui `createOrReplaceTempView` dan `spark.sql()`:

In [7]:
# Daftarkan DataFrame ke Temporary View
df_bersih.createOrReplaceTempView("transaksi")
df_cabang.createOrReplaceTempView("cabang")

# Jalankan kueri SQL
df_sql = spark.sql("""
    SELECT 
        c.kota,
        c.pic_cabang,
        COUNT(t.order_id) AS total_transaksi,
        SUM(t.total_pendapatan) AS total_omset
    FROM transaksi t
    INNER JOIN cabang c ON t.kota = c.kota
    GROUP BY c.kota, c.pic_cabang
    ORDER BY total_omset DESC
""")

print("Hasil Analisis Menggunakan Spark SQL:")
df_sql.show()

Hasil Analisis Menggunakan Spark SQL:


+----------+---------------+---------------+-----------+
|      kota|     pic_cabang|total_transaksi|total_omset|
+----------+---------------+---------------+-----------+
|  Magelang| Rani Handayani|              3|    2400000|
|  Semarang|     Sari Indah|              2|    2400000|
| Purworejo|Fitri Wulandari|              2|    2300000|
|Yogyakarta|    Joko Widodo|              2|    1875000|
|      Solo|   Bayu Nugroho|              1|     660000|
+----------+---------------+---------------+-----------+



### 7. Menyimpan Hasil Akhir (LOAD)
Menyimpan dataset hasil olahan dalam format **Apache Parquet** terpartisi berdasarkan kolom `kategori`:

In [8]:
output_dir = "demo_cheatsheet_parquet"

# Simpan ke format Parquet terpartisi kategori
df_bersih.write.mode("overwrite").partitionBy("kategori").parquet(output_dir)

print(f"✅ Data berhasil disimpan ke: {output_dir}")

# Verifikasi pembacaan ulang dari Parquet
df_cek = spark.read.parquet(output_dir)
print(f"Verifikasi: {df_cek.count()} baris berhasil dimuat kembali secara utuh.")

# Bersihkan direktori demo
!rm -rf demo_cheatsheet_parquet

✅ Data berhasil disimpan ke: demo_cheatsheet_parquet


Verifikasi: 10 baris berhasil dimuat kembali secara utuh.


### 8. Menutup SparkSession
Melepaskan alokasi thread CPU dan memori JVM:

In [9]:
spark.stop()
print("🛑 SparkSession berhasil dihentikan. RAM & CPU telah dibebaskan!")

🛑 SparkSession berhasil dihentikan. RAM & CPU telah dibebaskan!


---
## 💻 Terminal Cheatsheet: Perintah Dasar HDFS Wajib Hafal

| Perintah | Sintaks Lengkap | Penjelasan Sederhana |
|---|---|---|
| **Buat Folder** | `hdfs dfs -mkdir -p /user/$USER/raw` | Membuat folder secara berjenjang di HDFS |
| **Upload File** | `hdfs dfs -put -f file_lokal.csv /user/$USER/raw/` | Mengunggah file dari laptop ke HDFS (`-f` = overwrite jika sudah ada) |
| **Lihat Isi Folder** | `hdfs dfs -ls -h /user/$USER/raw/` | Melihat daftar file di HDFS beserta ukuran terbaca (KB/MB) |
| **Intip Isi File** | `hdfs dfs -cat /user/$USER/raw/file.csv \| head -n 10` | Menampilkan 10 baris pertama file dari HDFS ke terminal |
| **Download File** | `hdfs dfs -get -f /user/$USER/raw/file.csv ./` | Menyalin file dari HDFS turun ke disk laptop |
| **Cek Ukuran Disk** | `hdfs dfs -du -h /user/$USER/` | Melihat berapa megabyte ruang HDFS yang terpakai |
| **Hapus File/Folder** | `hdfs dfs -rm -r /user/$USER/raw/file.csv` | Menghapus file atau direktori di HDFS secara rekursif |

---
## 🎓 Jawaban Emas Pertanyaan Teori & Dosen (Gold Standard Answers)

### Q1: Apa bedanya Apache Hadoop vs Apache Spark?
> **Jawaban:** "Hadoop fokus utama pada penyimpanan file terdistribusi (*HDFS*) dan memiliki mesin proses batch klasik (*MapReduce*) yang lambat karena selalu menulis data perantara ke hard disk. Sedangkan **Apache Spark** adalah engine komputasi modern yang memproses data langsung di dalam memori RAM (*In-Memory Computing*), sehingga eksekusinya bisa 10 hingga 100 kali lebih cepat daripada Hadoop MapReduce."

### Q2: Jelaskan arsitektur HDFS! Apa fungsi NameNode dan DataNode?
> **Jawaban:** "HDFS menggunakan arsitektur Master-Worker:
> - **NameNode (Master):** Mengelola metadata direktori, nama file, hak akses, dan memetakan lokasi blok-blok data. NameNode **tidak** menyimpan isi data aktual.
> - **DataNode (Worker):** Menyimpan potongan blok data fisik sebenarnya (default 128 MB per blok) dan mengirim laporan kesehatan (*heartbeat*) secara berkala ke NameNode.
> - **Secondary NameNode:** Melakukan penggabungan berkala (*checkpointing*) antara file `fsimage` dan `edit logs` agar proses boot NameNode tidak memakan waktu lama, bukan sekadar pengganti instan saat NameNode mati."

### Q3: Mengapa kita memilih Parquet daripada CSV untuk Big Data?
> **Jawaban:** "Ada 4 keunggulan utama Parquet dibanding CSV:
> 1. **Columnar Storage:** Data disimpan per kolom, bukan per baris. Kueri analitik hanya membaca kolom yang dibutuhkan (*Column Pruning*), sehingga I/O disk sangat hemat.
> 2. **Kompresi Tinggi:** Parquet secara native terkompresi (biasanya Snappy), menghemat ukuran file 70% sampai 80% dibanding CSV.
> 3. **Menyimpan Skema:** Tipe data (integer, string, timestamp) tersimpan permanen di metadata file, tidak perlu proses `inferSchema` lambat seperti CSV.
> 4. **Predicate Pushdown:** Filter data (`WHERE / filter`) dapat dievaluasi langsung di level penyimpanan sebelum dimuat ke memori."

### Q4: Apa itu Lazy Evaluation di Apache Spark?
> **Jawaban:** "**Lazy Evaluation** adalah strategi Spark di mana transformasi data (seperti `filter`, `select`, `join`) tidak langsung dieksekusi saat kode ditulis. Spark hanya mencatat rencana operasi dalam bentuk graf (*DAG / Directed Acyclic Graph*). Eksekusi fisik baru benar-benar berjalan ketika ada perintah bertipe **Action** (seperti `show()`, `count()`, atau `write`). Ini memungkinkan Spark mengoptimasi alur kerja secara global lewat *Catalyst Optimizer* sebelum menyentuh data."

### Q5: Kapan kita menggunakan Window Function dibanding GroupBy?
> **Jawaban:** "`groupBy` akan merangkum dan **memadatkan baris** menjadi satu baris per grup (agregasi murni). Sebaliknya, **Window Function** melakukan perhitungan agregasi atau perankingan per kelompok baris **tanpa menghilangkan detail baris aslinya**, sehingga sangat cocok untuk kasus mencari ranking (Top-N), running total, atau perbandingan antarbaris."

### Q6: Apa perbedaan antara `row_number()`, `rank()`, dan `dense_rank()`?
> **Jawaban:**
> - `row_number()`: Memberikan nomor urut unik berurutan (1, 2, 3, 4) tanpa peduli ada nilai yang kembar.
> - `rank()`: Nilai yang nilainya sama mendapat ranking yang sama, namun urutan berikutnya akan melompat (misal ada ranking kembar 2, maka urutannya: 1, 2, 2, 4).
> - `dense_rank()`: Nilai kembar mendapat ranking sama tanpa lompatan urutan (misal: 1, 2, 2, 3)."

---
## 🛠️ Panduan Pertolongan Pertama (Troubleshooting Darurat)

| Masalah | Penyebab | Solusi Cepat |
|---|---|---|
| Perintah `hdfs` atau `start-dfs.sh` tidak ditemukan | Path belum terdaftar di session shell | Jalankan manual: `~/hadoop/sbin/start-dfs.sh` atau tambahkan path: `set -gx PATH ~/hadoop/bin ~/hadoop/sbin $PATH` |
| `NameNode` atau `DataNode` tidak muncul di `jps` | Service SSH belum jalan atau port macet | Jalankan `sudo systemctl start sshd`, lalu stop dan start ulang: `stop-dfs.sh && start-dfs.sh` |
| HDFS Safe Mode Error: `Cannot create directory... NameNode is in safe mode` | NameNode sedang melindungi diri saat startup | Paksa keluar dari safe mode: `hdfs dfsadmin -safemode leave` |
| PySpark Error: `Java gateway process exited before sending its port number` | Versi Java bentrok atau `JAVA_HOME` salah | Pastikan OpenJDK 11 terpasang dan tambahkan: `os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"` sebelum buat SparkSession |
| PySpark Error: `Cannot assign requested address` | Masalah binding localhost di Linux | Tambahkan: `os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"` |
| File Parquet tidak bisa ditimpa | File sedang dibuka atau mode default error | Selalu gunakan `.mode("overwrite")` sebelum `.parquet(...)` |